In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from itertools import combinations
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import wasserstein_distance, gaussian_kde

In [2]:
#Dataset
torch.random.manual_seed(42)
unlabeled = 20
num_classes = 2
label_budget = 4

data = torch.empty(unlabeled, num_classes).uniform_(-1, 1)
# Noise margin
margin = 0.0 # 0.0 means no noise, increase for more noise
distance = (data[:, 1] - data[:, 0]) / (2 ** 0.5)  # distance from the decision boundary (x=y)
labels = torch.where(
    distance.abs() < margin,
    torch.bernoulli(torch.full((unlabeled,), 0.5)),  # in margin
    (distance >= 0).float()                          # outside of margin
).unsqueeze(1)

all_subsets = list(combinations(range(unlabeled), label_budget))
x_all = torch.stack([data[list(subset)] for subset in all_subsets])
y_all = torch.stack([labels[list(subset)] for subset in all_subsets])

In [3]:
# x @ W.T + b
N = len(all_subsets)
D_in = num_classes
H = 0 # hidden layer size - not used
D_out = 1
W = torch.randn(N, D_out, D_in, requires_grad=True)
b = torch.randn(N, D_out, requires_grad=True)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD([W, b], lr=0.1)

# Training loop
num_epochs = 100
loss_history = []
for epoch in range(num_epochs):
    optimizer.zero_grad()
    logits = x_all @ W.transpose(-1, -2) + b.unsqueeze(1)
    loss = criterion(logits, y_all)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
print(f'Final Loss: {loss_history[-1]:.4f}')

Final Loss: 0.8676


In [4]:
#test dataset
test_data = torch.cartesian_prod(torch.linspace(-1, 1, 20), torch.linspace(-1, 1, 20))
test_labels = (test_data[:, 1] >= test_data[:, 0]).float().unsqueeze(1)
test_dataset = TensorDataset(test_data, test_labels)
test_loader = DataLoader(test_dataset, batch_size=400, shuffle=False)

# Evaluate test dataset
with torch.no_grad():
    for test_x, test_y in test_loader:
        test_logits = test_x.unsqueeze(0) @ W.transpose(-1, -2) + b.unsqueeze(1)
        test_preds = (test_logits >= 0).float()
        accuracies =(test_preds == test_y.unsqueeze(0)).float().mean(dim=[1, 2])
        
print(f'Test Accuracy: {accuracies.mean().item():.4f}')

Test Accuracy: 0.5000


In [5]:
# Visualization of NN accuracy distribution across all subsets (Ground Truth)
plt.figure(figsize=(18, 12))

# get hist return values: n (count), bins (edges), patches (bar objects)
n, bins, patches = plt.hist(accuracies.numpy(), bins=50, edgecolor='black', color='steelblue', alpha=0.5)

# get number of subests for each accuracy bin
for i in range(len(patches)):
    count = n[i]
    if count > 0: # only show number for bars with count > 0
        # get bar center x and height y
        x = patches[i].get_x() + patches[i].get_width() / 2
        y = patches[i].get_height()
        
        # show count above the bar, with a small offset (max(n)*0.01) to avoid overlap with the bar
        plt.text(x, y + (max(n)*0.01), str(int(count)), 
                 ha='center', va='bottom', fontsize=12, rotation=60)
        
plt.axvline(np.mean(accuracies.numpy()), color='blue', linestyle='--', label=f'Mean: {np.mean(accuracies.numpy()):.4f}')
plt.xlabel('Accuracy')
plt.ylabel('Number of Subsets')
plt.title(f'Distribution of Test Accuracy across all {len(all_subsets)} (n={unlabeled}, k={label_budget})Subsets w/ Counts')
plt.legend(fontsize=24)
plt.tight_layout()
plt.savefig(f'Distribution {len(all_subsets)}, n={unlabeled}, k={label_budget} with Counts, Noise={margin}.png')


In [ ]:
# Histogram Estimation of the accuracy distribution
true_accuracies = np.array(accuracies)
N_Total = len(true_accuracies)
sampling_rate = 0.10
sample_size = int(N_Total * sampling_rate)

np.random.seed(42) 
sampled_indices = np.random.choice(N_Total, size=sample_size, replace=False)
sampled_accuracies = true_accuracies[sampled_indices]
#print(f"Number of Ground Truth Subsets: {N_Total}")
#print(f"Number of Sampled Subsets: {len(sampled_accuracies)}")

# Wasserstein Distance of the two distributions
w_dist = wasserstein_distance(true_accuracies, sampled_accuracies)

# Visualize the sampled accuracy distribution
plt.figure(figsize=(12, 6))

# histogram of the true accuracy distribution using all subsets
plt.hist(true_accuracies, bins=50, range=(0, 1), density=True, 
         alpha=0.4, color='steelblue', label='Ground Truth (100% Subsets)', edgecolor='black')

# histogram estimate using 10% samples
plt.hist(sampled_accuracies, bins=50, range=(0, 1), density=True, 
         alpha=0.6, color='sandybrown', label=f'Histogram Estimate ({int(sampling_rate*100)}% Samples)', edgecolor='darkorange')

plt.title(f'Distribution Approximation using Histogram {sampling_rate:.2%}\nWasserstein Distance: {w_dist:.5f}', fontsize=14)
plt.xlabel('Accuracy')
plt.ylabel('Probability Density')
plt.legend(fontsize=12)
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(f'Approximation_Histogram_Baseline {sampling_rate:.2%}.png')
print("histogram estimate saved")


C:\Users\yeh75\AppData\Local\Temp\ipykernel_7808\3985172123.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  true_accuracies = np.array(accuracies)


histogram estimate saved


In [7]:
# Wasserstein distance between accuracy distribution and uniform distribution
np.random.seed(42)
uniform_dist = np.random.uniform(0, 1, size=N_Total)
wd = wasserstein_distance(accuracies.numpy(), uniform_dist)
print(f'Wasserstein Distance to Uniform Distribution: {wd:.4f}')

# Visualization of accuracy distribution vs uniform distribution
plt.figure(figsize=(12, 6))

# Accuracy Distribution of Ground Truth 
plt.hist(true_accuracies, bins=50, range=(0, 1), density=True, 
         alpha=0.5, color='steelblue', edgecolor='black', label='True Subset Performance Distribution')

# Uniform distribution
plt.hist(uniform_dist, bins=50, range=(0, 1), density=True, 
         alpha=0.3, color='sandybrown', edgecolor='dimgray', linestyle='--', label='Theoretical Uniform Distribution (Null Hypothesis)')

# Wasserstein Distance annotation
plt.title(f'Quantifying Non-Uniformity (H1 Verification)\nWasserstein Distance to Uniform Distribution: {wd:.4f}', fontsize=14, fontweight='bold')
plt.xlabel('Accuracy')
plt.ylabel('Probability Density')
plt.legend(fontsize=12, loc='upper left')
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig('H1_Verification_Uniform_Comparison.png')
print("H1 verification saved!")


# Wasserstein distance between accuracy distribution and Gaussian distribution
gaussian_dist = np.random.normal(0.5, 0.1, size=N_Total)
wd = wasserstein_distance(accuracies.numpy(), gaussian_dist)
print(f'Wasserstein Distance to Gaussian Distribution: {wd:.4f}')

# Visualization of accuracy distribution vs Gaussian distribution
plt.figure(figsize=(12, 6))

# Accuracy Distribution of Ground Truth 
plt.hist(true_accuracies, bins=50, range=(0, 1), density=True, 
         alpha=0.5, color='steelblue', edgecolor='black', label='True Subset Performance Distribution')

# Gaussian distribution
plt.hist(gaussian_dist, bins=50, range=(0, 1), density=True, 
         alpha=0.3, color='sandybrown', edgecolor='dimgray', linestyle='--', label='Theoretical Gaussian Distribution (Alternative Hypothesis)')

# Wasserstein Distance annotation
plt.title(f'Quantifying Non-Uniformity (H1 Verification)\nWasserstein Distance to Gaussian Distribution: {wd:.4f}', fontsize=14, fontweight='bold')
plt.xlabel('Accuracy')
plt.ylabel('Probability Density')
plt.legend(fontsize=12, loc='upper left')
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig('H1_Verification_Gaussian_Comparison.png')
print("Gaussian verification saved!")

Wasserstein Distance to Uniform Distribution: 0.1175
H1 verification saved!
Wasserstein Distance to Gaussian Distribution: 0.0563
Gaussian verification saved!


In [19]:
# KDE of accuracy distribution
# using guassian_kde to fit a KDE model to the sampled accuracies (10% samples)
bw_method = 'scott' 
kde_fit = gaussian_kde(sampled_accuracies, bw_method=bw_method) # use Scott's rule for bandwidth selection
np.random.seed(42)
kde_samp = kde_fit.resample(size=N_Total).flatten()
kde_samp = np.clip(kde_samp, 0, 1) # clip to [0, 1] since accuracy cannot be outside this range
w_dist_kde = wasserstein_distance(true_accuracies, kde_samp)

# evaluate the KDE model on a range of accuracy values from 0 to 1
x_evaluation = np.linspace(0, 1, 500)
kde_densities = kde_fit(x_evaluation) # get the estimated density values from the KDE model for the evaluation points

# visualize the KDE approximation compared to the ground truth distribution
plt.figure(figsize=(12, 6))

# Ground Truth distribution using all subsets
plt.hist(true_accuracies, bins=50, range=(0, 1), density=True, 
         alpha=0.3, color='steelblue', label='Ground Truth (100% Subsets)', edgecolor='black')

# KDE approximation using 10% samples
plt.plot(x_evaluation, kde_densities, color='crimson', lw=3, 
         label=f'KDE Estimate ({int(sampling_rate*100)}% Samples), \n (Bandwidth: {bw_method})')

plt.title(f'Distribution Approximation using Kernel Density Estimation (KDE)\nWasserstein Distance: {w_dist_kde:.5f}', fontsize=14, fontweight='bold')
plt.xlabel('Accuracy')
plt.ylabel('Probability Density')
plt.xlim(-0.05, 1.05)
plt.legend(fontsize=12)
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(f'Approximation_KDE_Baseline ({bw_method}, {int(sampling_rate*100)}%).png')
print("KDE Approximation saved!")

KDE Approximation saved!
